# YOLO11 NSFW Detection Training - Kaggle Notebook
## Safe Training with No Image Display

This notebook trains a YOLO11 model for NSFW body-part detection.
- **NEVER displays images** (NSFW safety)
- All outputs saved to `/kaggle/working/outputs`
- Final ZIP created for download


## 1. Setup: Download and Extract Dataset


In [1]:
# ============================
# Dataset configuration
# ============================

FILE_ID = "1XFTw-fmO6PWlsVv-6Hh5AkQeVhsY8wa5"   # Google Drive file ID
ARCHIVE_PATH = "/kaggle/working/final_balanced.7z"
DATA_DIR_SOURCE = "/kaggle/working/final_balanced"
# Use faster storage for training (Kaggle /kaggle/tmp is faster)
DATA_DIR = "/kaggle/tmp/final_balanced"        # Faster storage for training
OUT_DIR = "/kaggle/working/outputs"

import os
os.makedirs(DATA_DIR_SOURCE, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)

# ============================
# Download dataset from Google Drive (large file safe)
# ============================
print("Downloading dataset from Google Drive...")

%pip install -q gdown py7zr

import gdown

gdown.download(
    url=f"https://drive.google.com/uc?id={FILE_ID}",
    output=ARCHIVE_PATH,
    quiet=False,
    fuzzy=True,
)

print("\nDownload complete:", ARCHIVE_PATH)
print("File size (bytes):", os.path.getsize(ARCHIVE_PATH))

# ============================
# Extract dataset to source location
# ============================
import py7zr

print("\nExtracting dataset...")
with py7zr.SevenZipFile(ARCHIVE_PATH, mode="r") as z:
    z.extractall(path=DATA_DIR_SOURCE)

# ============================
# Copy to faster storage for training (if different)
# ============================
if DATA_DIR != DATA_DIR_SOURCE:
    print("\nCopying dataset to faster storage for training...")
    import shutil
    if os.path.exists(DATA_DIR):
        shutil.rmtree(DATA_DIR)
    shutil.copytree(DATA_DIR_SOURCE, DATA_DIR)
    print(f"✓ Dataset copied to fast storage: {DATA_DIR}")

print(f"\n✓ Dataset ready at: {DATA_DIR}")
print(f"✓ Output directory: {OUT_DIR}")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.7/69.7 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.5/99.5 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.3/141.3 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 428.8/428.8 kB 13.0 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


Downloading...
From (original): https://drive.google.com/uc?id=1XFTw-fmO6PWlsVv-6Hh5AkQeVhsY8wa5
From (redirected): https://drive.google.com/uc?id=1XFTw-fmO6PWlsVv-6Hh5AkQeVhsY8wa5&confirm=t&uuid=00fbb8e0-7aac-454e-a0d0-45b8f9c663ca
To: /kaggle/working/final_balanced.7z
100%|██████████| 7.26G/7.26G [00:59<00:00, 123MB/s] 



Download complete: /kaggle/working/final_balanced.7z
File size (bytes): 7263644913

Extracting dataset...

Copying dataset to faster storage for training...
✓ Dataset copied to fast storage: /kaggle/tmp/final_balanced

✓ Dataset ready at: /kaggle/tmp/final_balanced
✓ Output directory: /kaggle/working/outputs


In [2]:
import os

archive_path = "/kaggle/working/final_balanced.7z"

if os.path.exists(archive_path):
    os.remove(archive_path)
    print("✓ Deleted archive file:", archive_path)
else:
    print("Archive file not found.")


✓ Deleted archive file: /kaggle/working/final_balanced.7z


In [5]:
!pip install ultralytics
import os
import sys
import time
import warnings
import shutil
from pathlib import Path
from datetime import datetime
import yaml
import json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')  # Non-interactive backend - NO DISPLAY
import matplotlib.pyplot as plt

# Suppress warnings
warnings.filterwarnings("ignore")
os.environ["USE_LIBUV"] = "0"
os.environ["TORCH_USE_LIBUV"] = "0"
os.environ["TORCHELASTIC_USE_LIBUV"] = "0"

# Import YOLO
try:
    from ultralytics import YOLO
except ImportError:
    print("❌ Error: ultralytics not installed!")
    print("   Install with: pip install ultralytics")
    sys.exit(1)

import torch
import torch.nn as nn

print("✓ All imports successful")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 4.7 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of mkl-fft to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of mkl-random to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of mkl-umath to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 23.4 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.8/16.8 MB 89.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 5.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 106.5 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 78.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [6]:
# Configuration - Optimized for T4 16GB GPU(s) on Kaggle
# Auto-detect number of GPUs and configure accordingly
# NOTE: On Kaggle (Linux), DDP works perfectly - this is the recommended approach
import torch
num_gpus = torch.cuda.device_count() if torch.cuda.is_available() else 0

# Set device: use all available GPUs if multiple, otherwise single GPU
if num_gpus >= 2:
    device_config = [0, 1]  # Use both GPUs (YOLO automatically uses DDP for multi-GPU)
    print(f"✓ Detected {num_gpus} GPUs - Using multi-GPU training (GPUs: {device_config})")
    print(f"  ✓ DDP (Distributed Data Parallel) works perfectly on Kaggle/Linux")
    print(f"  ✓ YOLO handles multi-GPU automatically - no manual setup needed")
    print(f"  ✓ Expected speedup: ~{num_gpus}x faster training")
elif num_gpus == 1:
    device_config = 0  # Single GPU
    print(f"✓ Detected 1 GPU - Using single GPU training")
else:
    device_config = 'cpu'
    print(f"⚠ No GPUs detected - Using CPU (very slow!)")

MODEL_CONFIG = {
    'model': 'yolo11m-seg.pt',  # Segmentation model
    'task': 'segment',  # Segmentation (masks + boxes)
    'imgsz': 768,  # Image size
    'batch': 32,  # Batch size per GPU (total batch = batch * num_gpus) - Increased from 16 to utilize more GPU memory
    'epochs': 100,  # Number of epochs
    'patience': 15,  # Early stopping patience
    'device': device_config,  # Auto-configured: [0, 1] for multi-GPU, 0 for single GPU
    'workers': 4,  # Data loading workers per GPU
    'optimizer': 'SGD',  # Optimizer
    'lr0': 0.016,  # Initial learning rate
    'lrf': 0.1,  # Final learning rate factor
    'momentum': 0.9,  # SGD momentum
    'weight_decay': 0.0005,  # Weight decay
    'warmup_epochs': 5,  # Warmup epochs
    'warmup_momentum': 0.8,  # Warmup momentum
    'warmup_bias_lr': 0.1,  # Warmup bias learning rate
    'box': 7.5,  # Box loss gain
    'cls': 0.5,  # Class loss gain
    'dfl': 1.5,  # DFL loss gain
    'hsv_h': 0.015,  # HSV-Hue augmentation
    'hsv_s': 0.7,  # HSV-Saturation augmentation
    'hsv_v': 0.4,  # HSV-Value augmentation
    'degrees': 0.0,  # Rotation augmentation
    'translate': 0.1,  # Translation augmentation
    'scale': 0.5,  # Scale augmentation
    'shear': 0.0,  # Shear augmentation
    'perspective': 0.0,  # Perspective augmentation
    'flipud': 0.0,  # Vertical flip probability
    'fliplr': 0.5,  # Horizontal flip probability
    'mosaic': 1.0,  # Mosaic augmentation probability
    'mixup': 0.0,  # Mixup augmentation probability
    'copy_paste': 0.0,  # Copy-paste augmentation probability
}

# Class names
CLASS_NAMES = {
    0: 'anus',
    1: 'breast',
    2: 'female_genital',
    3: 'male_genital'
}

# Dataset paths - ensure DATA_DIR is defined
if 'DATA_DIR' not in globals():
    DATA_DIR = "/kaggle/working/final_balanced/final_balanced"
    print(f"⚠ DATA_DIR not defined, using default: {DATA_DIR}")

DATASET_YAML = f"{DATA_DIR}/dataset.yaml"

print("✓ Configuration loaded")
print(f"  DATA_DIR: {DATA_DIR}")
print(f"  DATASET_YAML: {DATASET_YAML}")


✓ Detected 2 GPUs - Using multi-GPU training (GPUs: [0, 1])
  ✓ DDP (Distributed Data Parallel) works perfectly on Kaggle/Linux
  ✓ YOLO handles multi-GPU automatically - no manual setup needed
  ✓ Expected speedup: ~2x faster training
✓ Configuration loaded
  DATA_DIR: /kaggle/tmp/final_balanced
  DATASET_YAML: /kaggle/tmp/final_balanced/dataset.yaml


## 3. Utility Functions


In [7]:
def check_dataset_structure(dataset_dir: str):
    """Check and verify dataset structure."""
    stats = {
        'exists': False,
        'yaml_exists': False,
        'splits': {},
        'total_images': 0,
        'total_labels': 0
    }
    
    dataset_path = Path(dataset_dir)
    if not dataset_path.exists():
        print(f"❌ Dataset directory not found: {dataset_dir}")
        return stats
    
    stats['exists'] = True
    
    # Check YAML file
    yaml_path = dataset_path / "dataset.yaml"
    if yaml_path.exists():
        stats['yaml_exists'] = True
        with open(yaml_path, 'r') as f:
            yaml_data = yaml.safe_load(f)
            stats['yaml_data'] = yaml_data
    
    # Check splits
    for split in ['train', 'val', 'test']:
        images_dir = dataset_path / "images" / split
        labels_dir = dataset_path / "labels" / split
        
        split_stats = {
            'images_exist': images_dir.exists(),
            'labels_exist': labels_dir.exists(),
            'image_count': 0,
            'label_count': 0
        }
        
        if images_dir.exists():
            image_extensions = ['.jpg', '.jpeg', '.png', '.JPG', '.JPEG', '.PNG']
            for ext in image_extensions:
                split_stats['image_count'] += len(list(images_dir.glob(f"*{ext}")))
        
        if labels_dir.exists():
            split_stats['label_count'] += len(list(labels_dir.glob("*.txt")))
        
        stats['splits'][split] = split_stats
        stats['total_images'] += split_stats['image_count']
        stats['total_labels'] += split_stats['label_count']
    
    return stats


def print_dataset_info(stats):
    """Print dataset information."""
    print("=" * 80)
    print("Dataset Structure Check")
    print("=" * 80)
    
    if not stats['exists']:
        print("❌ Dataset directory does not exist!")
        return
    
    print(f"✓ Dataset directory exists")
    print(f"✓ YAML file exists: {stats['yaml_exists']}")
    
    if stats['yaml_exists']:
        yaml_data = stats.get('yaml_data', {})
        print(f"  Classes: {yaml_data.get('nc', 'N/A')}")
    
    print("\nSplits:")
    for split, split_stats in stats['splits'].items():
        print(f"  {split.upper()}: {split_stats['image_count']:,} images, {split_stats['label_count']:,} labels")
    
    print(f"\nTotal: {stats['total_images']:,} images, {stats['total_labels']:,} labels")


def check_gpu_availability():
    """Check GPU availability."""
    info = {
        'cuda_available': torch.cuda.is_available(),
        'device_count': 0,
        'devices': []
    }
    
    if torch.cuda.is_available():
        info['device_count'] = torch.cuda.device_count()
        for i in range(torch.cuda.device_count()):
            device_info = {
                'id': i,
                'name': torch.cuda.get_device_name(i),
                'memory_total_gb': torch.cuda.get_device_properties(i).total_memory / (1024**3)
            }
            info['devices'].append(device_info)
    
    return info


def print_gpu_info(info):
    """Print GPU information."""
    print("=" * 80)
    print("GPU Information")
    print("=" * 80)
    
    if not info['cuda_available']:
        print("❌ CUDA not available. Training will use CPU (very slow!)")
        return
    
    print(f"✓ CUDA available")
    print(f"✓ Number of GPUs: {info['device_count']}")
    
    for device in info['devices']:
        print(f"GPU {device['id']}: {device['name']}")
        print(f"  Total Memory: {device['memory_total_gb']:.2f} GB")


def format_time(seconds: float) -> str:
    """Format time in seconds to human-readable string."""
    if seconds < 60:
        return f"{seconds:.1f}s"
    elif seconds < 3600:
        return f"{seconds/60:.1f}m"
    else:
        hours = int(seconds // 3600)
        minutes = int((seconds % 3600) // 60)
        return f"{hours}h {minutes}m"

print("✓ Utility functions defined")


✓ Utility functions defined


## 4. Verify Dataset and GPU


In [8]:
# Verify dataset
print("=" * 100)
print(" Verifying Dataset ".center(100, "="))
print("=" * 100)
print()

# Ensure DATA_DIR is defined (fallback if cell 2 wasn't run)
if 'DATA_DIR' not in globals():
    DATA_DIR = "/kaggle/working/final_balanced/final_balanced"
    print(f"⚠ DATA_DIR not defined, using default: {DATA_DIR}")

# CRITICAL: Re-detect the correct path if dataset.yaml is not found
print(f"Current DATA_DIR: {DATA_DIR}")
dataset_path = Path(DATA_DIR)
yaml_path = dataset_path / "dataset.yaml"

if not yaml_path.exists():
    print(f"\n⚠ dataset.yaml not found at {yaml_path}")
    print("  Searching for correct path...")
    
    # Search for dataset.yaml in nested structure
    search_paths = [
        Path(DATA_DIR),
        Path("/kaggle/working/final_balanced/final_balanced"),
        Path("/kaggle/working/final_balanced") / "final_balanced",
        Path("/kaggle/working/final_balanced"),
    ]
    
    for search_path in search_paths:
        test_yaml = search_path / "dataset.yaml"
        if test_yaml.exists():
            DATA_DIR = str(search_path)
            # CRITICAL: Update DATASET_YAML when DATA_DIR is updated
            DATASET_YAML = f"{DATA_DIR}/dataset.yaml"
            print(f"✓ Found dataset.yaml at: {test_yaml}")
            print(f"✓ Updated DATA_DIR to: {DATA_DIR}")
            print(f"✓ Updated DATASET_YAML to: {DATASET_YAML}")
            dataset_path = Path(DATA_DIR)
            yaml_path = test_yaml
            break
    else:
        print("❌ Could not find dataset.yaml in any expected location")

# Debug: Check what path we're using and verify it exists
print(f"\nVerifying dataset at: {DATA_DIR}")
print(f"Path exists: {dataset_path.exists()}")

if dataset_path.exists():
    print(f"\nContents of {DATA_DIR}:")
    try:
        contents = list(dataset_path.iterdir())
        for item in contents[:10]:  # Show first 10 items
            print(f"  - {item.name} ({'dir' if item.is_dir() else 'file'})")
        if len(contents) > 10:
            print(f"  ... and {len(contents) - 10} more items")
    except Exception as e:
        print(f"  Error listing contents: {e}")
    
    # Check for dataset.yaml
    yaml_path = dataset_path / "dataset.yaml"
    print(f"\ndataset.yaml exists: {yaml_path.exists()}")
    if yaml_path.exists():
        print(f"  Full path: {yaml_path}")
    else:
        print(f"  ❌ NOT FOUND at: {yaml_path}")
    
    # Check for images directory
    images_path = dataset_path / "images"
    print(f"images/ directory exists: {images_path.exists()}")
    if images_path.exists():
        print(f"  Full path: {images_path}")
        try:
            subdirs = [d.name for d in images_path.iterdir() if d.is_dir()]
            print(f"  Subdirectories: {subdirs}")
            # Count images in each subdirectory
            for subdir in subdirs:
                subdir_path = images_path / subdir
                img_count = len(list(subdir_path.glob("*.jpg"))) + len(list(subdir_path.glob("*.png")))
                print(f"    {subdir}: {img_count} images")
        except Exception as e:
            print(f"  Error checking subdirectories: {e}")
    else:
        print(f"  ❌ NOT FOUND at: {images_path}")

print()

dataset_stats = check_dataset_structure(DATA_DIR)
print_dataset_info(dataset_stats)

if not dataset_stats['exists']:
    print("\n❌ Dataset directory does not exist!")
    sys.exit(1)

if not dataset_stats['yaml_exists']:
    print("\n❌ dataset.yaml not found!")
    sys.exit(1)

# Check if splits have images
for split in ['train', 'val']:
    split_stats = dataset_stats['splits'].get(split, {})
    if split_stats.get('image_count', 0) == 0:
        print(f"\n❌ No images found in {split} split!")
        sys.exit(1)

print("\n✓ Dataset verified successfully!")


======================================== Verifying Dataset =========================================

Current DATA_DIR: /kaggle/tmp/final_balanced

⚠ dataset.yaml not found at /kaggle/tmp/final_balanced/dataset.yaml
  Searching for correct path...
✓ Found dataset.yaml at: /kaggle/working/final_balanced/final_balanced/dataset.yaml
✓ Updated DATA_DIR to: /kaggle/working/final_balanced/final_balanced
✓ Updated DATASET_YAML to: /kaggle/working/final_balanced/final_balanced/dataset.yaml

Verifying dataset at: /kaggle/working/final_balanced/final_balanced
Path exists: True

Contents of /kaggle/working/final_balanced/final_balanced:
  - dataset.yaml (file)
  - labels (dir)
  - images (dir)

dataset.yaml exists: True
  Full path: /kaggle/working/final_balanced/final_balanced/dataset.yaml
images/ directory exists: True
  Full path: /kaggle/working/final_balanced/final_balanced/images
  Subdirectories: ['test', 'val', 'train']
    test: 3160 images
    val: 4012 images
    train: 31918 images

D

In [9]:
# Fix NumPy / Matplotlib compatibility
%pip install "numpy<2" --force-reinstall -q

import numpy as np
print("NumPy version:", np.__version__)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 81.2 MB/s eta 0:00:00:00:0100:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.12.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
datasets 4.4.1 requires pyarrow>=21.0.0, but you have pyarrow 19.0.1 which is incompatible.
cesium 0.12.4 requires numpy<3.0,>=2.0, but you have numpy 1.26.4 which is incompatible.
google-colab 1.0.0 requires notebook==6.5.7, but you have notebook 6.5.4 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.2.3 which is incompatible.
google-colab 1.0.0 requires requests==2.32.3, but you have requests 2.32.5 which is incompatible.
google-colab 1.0.0 requires tornado==6.4.2, but you have tornado 6.5.2 which is incompatible

In [10]:
# Verify GPU
print("\n" + "=" * 100)
print(" Verifying GPU ".center(100, "="))
print("=" * 100)
print()

gpu_info = check_gpu_availability()
print_gpu_info(gpu_info)

if not gpu_info['cuda_available']:
    print("\n⚠ CUDA not available! Training will be very slow on CPU.")

print("\n✓ GPU check complete")



========================================== Verifying GPU ===========================================

GPU Information
✓ CUDA available
✓ Number of GPUs: 2
GPU 0: Tesla T4
  Total Memory: 14.74 GB
GPU 1: Tesla T4
  Total Memory: 14.74 GB

✓ GPU check complete


## 5. Load Model


In [11]:
print("=" * 100)
print(" Loading Model ".center(100, "="))
print("=" * 100)
print()

try:
    model = YOLO(MODEL_CONFIG['model'])
    print(f"✓ Model loaded: {MODEL_CONFIG['model']}")
    print(f"  Model type: {type(model).__name__}")
    print(f"  Task: {MODEL_CONFIG['task']}")
    print()
except Exception as e:
    print(f"❌ Failed to load model: {e}")
    import traceback
    traceback.print_exc()
    sys.exit(1)


========================================== Loading Model ===========================================

✓ Model loaded: yolo11m-seg.pt
  Model type: YOLO
  Task: segment



## 6. Training


In [12]:
# ================== PRETRAINED TRAINING BLOCK ==================
import os, time, yaml, torch
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from ultralytics import YOLO

print("===== PRETRAINED YOLO11m-seg TRAINING START =====")

# ----------------------------------------------------
# 0) OUTPUT DIR (for plots etc.)
# ----------------------------------------------------
if "OUT_DIR" not in globals():
    OUT_DIR = "/kaggle/working/outputs"
os.makedirs(OUT_DIR, exist_ok=True)
print(f"OUT_DIR: {OUT_DIR}")

# ----------------------------------------------------
# 1) PATHS
# ----------------------------------------------------
DATASET_YAML = "/kaggle/working/final_balanced/final_balanced/dataset.yaml"
CKPT_PATH    = "/kaggle/input/model-last/other/default/1/last (1).pt"

print(f"✓ Using dataset YAML: {DATASET_YAML}")
print(f"✓ Using pretrained checkpoint: {CKPT_PATH}")

# ----------------------------------------------------
# 2) FIX DATASET YAML PATHS
# ----------------------------------------------------
dataset_yaml_path = Path(DATASET_YAML)
if not dataset_yaml_path.exists():
    raise FileNotFoundError(f"dataset.yaml not found at {DATASET_YAML}")

base = str(dataset_yaml_path.parent)  # /kaggle/working/final_balanced/final_balanced
with open(dataset_yaml_path, "r") as f:
    data_yaml = yaml.safe_load(f)

data_yaml["path"]  = base
data_yaml["train"] = f"{base}/images/train"
data_yaml["val"]   = f"{base}/images/val"
data_yaml["test"]  = f"{base}/images/test"

with open(dataset_yaml_path, "w") as f:
    yaml.dump(data_yaml, f, sort_keys=False)

print("✓ Dataset YAML corrected")

# ----------------------------------------------------
# 3) LOAD CHECKPOINT (CPU) & EXTRACT HYPERPARAMS
# ----------------------------------------------------
import numpy as np

ckpt = torch.load(CKPT_PATH, map_location="cpu", weights_only=False)
train_args_old = ckpt["train_args"]
opt_old        = ckpt["optimizer"]

# --- normalize numpy scalars to native Python types ---
def to_py(v):
    if isinstance(v, (np.floating, np.integer)):
        return v.item()
    return v

for k, v in list(train_args_old.items()):
    train_args_old[k] = to_py(v)

pg = opt_old["param_groups"]
for g in pg:
    for k, v in list(g.items()):
        g[k] = to_py(v)

# --- now extract hyperparams using CLEANED values ---
lr0          = pg[0]["lr"]
momentum     = pg[0]["momentum"]
weight_decay = pg[1].get("weight_decay", 0.0005)

warmup_epochs   = train_args_old["warmup_epochs"]
warmup_momentum = train_args_old["warmup_momentum"]
warmup_bias_lr  = train_args_old["warmup_bias_lr"]

mask_ratio_old  = float(train_args_old.get("mask_ratio", 4.0))

print("\n======= Extracted Hyperparameters from Checkpoint =======")
print("lr0:", lr0)
print("momentum:", momentum)
print("weight_decay:", weight_decay)
print("warmup_epochs:", warmup_epochs)
print("warmup_momentum:", warmup_momentum)
print("warmup_bias_lr:", warmup_bias_lr)
print("mask_ratio (from ckpt):", mask_ratio_old, "-> using int:", int(mask_ratio_old))
print("==========================================================\n")

# --------------------------------------------------------------

print("\n======= Extracted Hyperparameters from Checkpoint =======")
print("lr0:", lr0)
print("momentum:", momentum)
print("weight_decay:", weight_decay)
print("warmup_epochs:", warmup_epochs)
print("warmup_momentum:", warmup_momentum)
print("warmup_bias_lr:", warmup_bias_lr)
print("mask_ratio (from ckpt):", mask_ratio_old, "-> using int:", int(mask_ratio_old))
print("==========================================================\n")

# ----------------------------------------------------
# 4) DEVICE CONFIG (BOTH GPUS IF AVAILABLE)
# ----------------------------------------------------
if torch.cuda.is_available():
    n_gpu = torch.cuda.device_count()
    if n_gpu >= 2:
        device_arg = "0,1"
    else:
        device_arg = 0
    print(f"✓ CUDA available, using device={device_arg}")
else:
    device_arg = "cpu"
    print("⚠ CUDA not available, using CPU (very slow)")

# ----------------------------------------------------
# 5) DEFINE NEW RUN DIR
# ----------------------------------------------------
run_root = Path("/kaggle/working/runs")
run_root.mkdir(parents=True, exist_ok=True)
run_name = f"yolo11m_seg_pretrained_exact_{int(time.time())}"
run_dir  = run_root / run_name

print(f"Run directory will be: {run_dir}")

# ----------------------------------------------------
# 6) LOAD MODEL WITH PRETRAINED WEIGHTS (NO RESUME)
# ----------------------------------------------------
# This loads architecture + weights, but does NOT resume optimizer/EMA
model = YOLO(CKPT_PATH)
print("✓ Loaded YOLO model with your pretrained weights\n")

# ----------------------------------------------------
# 7) BUILD TRAIN ARGS (SAME HYPERPARAMS, NO 'ema')
# ----------------------------------------------------
train_args = {
    "data": DATASET_YAML,
    "epochs": train_args_old.get("epochs", 100),
    "batch": train_args_old.get("batch", 32),
    "imgsz": train_args_old.get("imgsz", 768),

    "device": device_arg,
    "workers": train_args_old.get("workers", 4),

    "optimizer": "SGD",
    "lr0": lr0,
    "lrf": train_args_old["lrf"],
    "momentum": momentum,
    "weight_decay": weight_decay,

    "warmup_epochs": warmup_epochs,
    "warmup_momentum": warmup_momentum,
    "warmup_bias_lr": warmup_bias_lr,

    "box": train_args_old["box"],
    "cls": train_args_old["cls"],
    "dfl": train_args_old["dfl"],

    "hsv_h": train_args_old["hsv_h"],
    "hsv_s": train_args_old["hsv_s"],
    "hsv_v": train_args_old["hsv_v"],
    "degrees": train_args_old["degrees"],
    "translate": train_args_old["translate"],
    "scale": train_args_old["scale"],
    "shear": train_args_old["shear"],
    "perspective": train_args_old["perspective"],
    "flipud": train_args_old["flipud"],
    "fliplr": train_args_old["fliplr"],
    "mosaic": train_args_old["mosaic"],
    "mixup": train_args_old["mixup"],
    "cutmix": train_args_old.get("cutmix", 0.0),
    "copy_paste": train_args_old["copy_paste"],
    "copy_paste_mode": train_args_old.get("copy_paste_mode", "flip"),
    "auto_augment": train_args_old.get("auto_augment", "randaugment"),
    "erasing": train_args_old.get("erasing", 0.4),

    # mask_ratio must be INT now
    "mask_ratio": int(mask_ratio_old),

    "project": str(run_root),
    "name": run_name,
    "resume": False,          # IMPORTANT: not resuming
    "plots": False,            # YOLO will auto-save plots
    "save": True,             # save best.pt / last.pt
}

# ----------------------------------------------------
# 8) TRAIN (FULL PROGRESS BAR)
# ----------------------------------------------------
print("\n=========== TRAINING STARTED (FULL PROGRESS BAR) ===========\n")
results = model.train(**train_args)
print("\n=========== TRAINING COMPLETE ===========\n")
print(f"Run directory: {run_dir}")

# ----------------------------------------------------
# 9) EXTRA: SAVE TRAINING CURVES TO OUT_DIR
# ----------------------------------------------------
results_csv = run_dir / "results.csv"
if results_csv.exists():
    print(f"✓ Found results.csv: {results_csv}")
    df = pd.read_csv(results_csv)

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    # Box loss
    if "train/box_loss" in df.columns and "val/box_loss" in df.columns:
        axes[0, 0].plot(df["epoch"], df["train/box_loss"], label="train box")
        axes[0, 0].plot(df["epoch"], df["val/box_loss"], label="val box")
        axes[0, 0].set_title("Box loss")
        axes[0, 0].legend()
        axes[0, 0].grid(True)

    # Class loss
    if "train/cls_loss" in df.columns and "val/cls_loss" in df.columns:
        axes[0, 1].plot(df["epoch"], df["train/cls_loss"], label="train cls")
        axes[0, 1].plot(df["epoch"], df["val/cls_loss"], label="val cls")
        axes[0, 1].set_title("Class loss")
        axes[0, 1].legend()
        axes[0, 1].grid(True)

    # mAP50
    if "metrics/mAP50(B)" in df.columns:
        axes[1, 0].plot(df["epoch"], df["metrics/mAP50(B)"], label="mAP50")
        axes[1, 0].set_title("mAP@0.50")
        axes[1, 0].legend()
        axes[1, 0].grid(True)

    # mAP50-95
    if "metrics/mAP50-95(B)" in df.columns:
        axes[1, 1].plot(df["epoch"], df["metrics/mAP50-95(B)"], label="mAP50-95")
        axes[1, 1].set_title("mAP@0.50:0.95")
        axes[1, 1].legend()
        axes[1, 1].grid(True)

    plt.tight_layout()
    plot_path = Path(OUT_DIR) / "training_curves.png"
    plt.savefig(plot_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"✓ Training curves saved to: {plot_path}")
else:
    print("⚠ results.csv not found, cannot plot curves")

print("\n===== DONE. best.pt / last.pt are in:", run_dir / 'weights', "=====")


===== PRETRAINED YOLO11m-seg TRAINING START =====
OUT_DIR: /kaggle/working/outputs
✓ Using dataset YAML: /kaggle/working/final_balanced/final_balanced/dataset.yaml
✓ Using pretrained checkpoint: /kaggle/input/model-last/other/default/1/last (1).pt
✓ Dataset YAML corrected

======= Extracted Hyperparameters from Checkpoint =======
lr0: 0.012544000000000001
momentum: 0.9
weight_decay: 0.0005
warmup_epochs: 5
warmup_momentum: 0.8
warmup_bias_lr: 0.1
mask_ratio (from ckpt): 4.0 -> using int: 4


======= Extracted Hyperparameters from Checkpoint =======
lr0: 0.012544000000000001
momentum: 0.9
weight_decay: 0.0005
warmup_epochs: 5
warmup_momentum: 0.8
warmup_bias_lr: 0.1
mask_ratio (from ckpt): 4.0 -> using int: 4

✓ CUDA available, using device=0,1
Run directory will be: /kaggle/working/runs/yolo11m_seg_pretrained_exact_1764499597
✓ Loaded YOLO model with your pretrained weights


=========== TRAINING STARTED (FULL PROGRESS BAR) ===========

Ultralytics 8.3.233 🚀 Python-3.11.13 torch-2.6.0+

W1130 18:08:13.763000 132 torch/distributed/elastic/agent/server/api.py:719] Received 2 death signal, shutting down workers
W1130 18:08:13.763000 132 torch/distributed/elastic/multiprocessing/api.py:897] Sending process 134 closing signal SIGINT
W1130 18:08:13.766000 132 torch/distributed/elastic/multiprocessing/api.py:897] Sending process 135 closing signal SIGINT
[rank1]: Traceback (most recent call last):
[rank1]:   File "/root/.config/Ultralytics/DDP/_temp_mownx2r1135802302013840.py", line 13, in <module>
[rank1]:     results = trainer.train()
[rank1]:               ^^^^^^^^^^^^^^^
[rank1]:   File "/usr/local/lib/python3.11/dist-packages/ultralytics/engine/trainer.py", line 243, in train
[rank1]:     self._do_train()
[rank1]:   File "/usr/local/lib/python3.11/dist-packages/ultralytics/engine/trainer.py", line 427, in _do_train
[rank1]:     loss, self.loss_items = self.model(batch)
[rank1]:                             ^^^^^^^^^^^^^^^^^
[rank1]:   File "/usr/local/lib/python3.11/dist

     21/100      14.1G      1.039      1.448     0.6024      1.065         44        768: 2% ──────────── 24/998 1.2it/s 31.3s<13:01


KeyboardInterrupt: 

## 7. Export Model to Various Formats


In [13]:
# Export trained model to various formats for deployment
print("=" * 100)
print(" Exporting Model ".center(100, "="))
print("=" * 100)
print()

# Load best model for export
best_model_path = run_dir / "weights" / "best.pt"
if not best_model_path.exists():
    print("⚠ Best model weights not found, skipping export")
else:
    print(f"Loading best model: {best_model_path}")
    best_model = YOLO(str(best_model_path))
    
    # Create export directory
    export_dir = Path(OUT_DIR) / "exports"
    export_dir.mkdir(parents=True, exist_ok=True)
    
    # Export formats
    export_formats = {
        'onnx': {'format': 'onnx', 'imgsz': MODEL_CONFIG['imgsz'], 'simplify': True},
        'torchscript': {'format': 'torchscript', 'imgsz': MODEL_CONFIG['imgsz']},
        'engine': {'format': 'engine', 'imgsz': MODEL_CONFIG['imgsz']},  # TensorRT (if available)
    }
    
    exported_files = []
    
    for format_name, export_args in export_formats.items():
        try:
            print(f"\nExporting to {format_name.upper()}...")
            export_path = best_model.export(
                **export_args,
                project=str(export_dir),
                name=format_name
            )
            
            # Copy exported file to OUT_DIR root for easy access
            if isinstance(export_path, (str, Path)):
                export_path = Path(export_path)
                if export_path.exists():
                    # Find the actual exported file
                    exported_file = None
                    if export_path.is_file():
                        exported_file = export_path
                    else:
                        # Look for exported files in the directory
                        for ext in ['.onnx', '.torchscript', '.engine']:
                            found = list(export_path.parent.glob(f"*{ext}"))
                            if found:
                                exported_file = found[0]
                                break
                    
                    if exported_file:
                        # Copy to OUT_DIR root
                        dest_file = Path(OUT_DIR) / f"best_model.{format_name}"
                        if exported_file.suffix:
                            dest_file = Path(OUT_DIR) / f"best_model{exported_file.suffix}"
                        shutil.copy(exported_file, dest_file)
                        exported_files.append(dest_file)
                        print(f"  ✓ Exported to: {dest_file}")
            
        except Exception as e:
            print(f"  ⚠ Failed to export to {format_name}: {e}")
            # Continue with other formats
    
    if exported_files:
        print(f"\n✓ Successfully exported {len(exported_files)} model format(s)")
        print("  Exported formats:")
        for f in exported_files:
            print(f"    - {f.name}")
    else:
        print("\n⚠ No models were successfully exported")
    
    print()
print("=" * 100)
print(" Exporting Model ".center(100, "="))
print("=" * 100)
print()

# Load best model for export
best_model_path = run_dir / "weights" / "best.pt"
if not best_model_path.exists():
    print("⚠ Best model weights not found, skipping export")
else:
    print(f"Loading best model: {best_model_path}")
    best_model = YOLO(str(best_model_path))
    
    # Create export directory
    export_dir = Path(OUT_DIR) / "exports"
    export_dir.mkdir(parents=True, exist_ok=True)
    
    # Export formats
    export_formats = {
        'onnx': {'format': 'onnx', 'imgsz': MODEL_CONFIG['imgsz'], 'simplify': True},
        'torchscript': {'format': 'torchscript', 'imgsz': MODEL_CONFIG['imgsz']},
        'engine': {'format': 'engine', 'imgsz': MODEL_CONFIG['imgsz']},  # TensorRT (if available)
    }
    
    exported_files = []
    
    for format_name, export_args in export_formats.items():
        try:
            print(f"\nExporting to {format_name.upper()}...")
            export_path = best_model.export(
                **export_args,
                project=str(export_dir),
                name=format_name
            )
            
            # Copy exported file to OUT_DIR root for easy access
            if isinstance(export_path, (str, Path)):
                export_path = Path(export_path)
                if export_path.exists():
                    # Find the actual exported file
                    exported_file = None
                    if export_path.is_file():
                        exported_file = export_path
                    else:
                        # Look for exported files in the directory
                        for ext in ['.onnx', '.torchscript', '.engine']:
                            found = list(export_path.parent.glob(f"*{ext}"))
                            if found:
                                exported_file = found[0]
                                break
                    
                    if exported_file:
                        # Copy to OUT_DIR root
                        dest_file = Path(OUT_DIR) / f"best_model.{format_name}"
                        if exported_file.suffix:
                            dest_file = Path(OUT_DIR) / f"best_model{exported_file.suffix}"
                        shutil.copy(exported_file, dest_file)
                        exported_files.append(dest_file)
                        print(f"  ✓ Exported to: {dest_file}")
            
        except Exception as e:
            print(f"  ⚠ Failed to export to {format_name}: {e}")
            # Continue with other formats
    
    if exported_files:
        print(f"\n✓ Successfully exported {len(exported_files)} model format(s)")
        print("  Exported formats:")
        for f in exported_files:
            print(f"    - {f.name}")
    else:
        print("\n⚠ No models were successfully exported")
    
    print()


========================================= Exporting Model ==========================================

Loading best model: /kaggle/working/runs/yolo11m_seg_pretrained_exact_1764499597/weights/best.pt

Exporting to ONNX...
Ultralytics 8.3.233 🚀 Python-3.11.13 torch-2.6.0+cu124 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/
YOLO11m-seg summary (fused): 138 layers, 22,338,396 parameters, 0 gradients, 112.9 GFLOPs

PyTorch: starting from '/kaggle/working/runs/yolo11m_seg_pretrained_exact_1764499597/weights/best.pt' with input shape (1, 3, 768, 768) BCHW and output shape(s) ((1, 40, 12096), (1, 32, 192, 192)) (85.9 MB)
requirements: Ultralytics requirements ['onnxslim>=0.1.71', 'onnxruntime-gpu'] not found, attempting AutoUpdate...
Using Python 3.11.13 environment at: /usr
Resolved 14 packages in 253ms
Prepared 4 packages in 3.13s
Installed 4 packages in 12ms
 + color

## 8. Generate Training Curves (Saved to File Only)


In [14]:
# Read results.csv if it exists
results_csv = run_dir / "results.csv"
if results_csv.exists():
    print("Generating training curves...")
    
    # Read CSV
    df = pd.read_csv(results_csv)
    
    # Plot loss curves
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    # Train/Val Loss
    if 'train/box_loss' in df.columns and 'val/box_loss' in df.columns:
        axes[0, 0].plot(df['epoch'], df['train/box_loss'], label='Train Box Loss', color='blue')
        axes[0, 0].plot(df['epoch'], df['val/box_loss'], label='Val Box Loss', color='red')
        axes[0, 0].set_xlabel('Epoch')
        axes[0, 0].set_ylabel('Loss')
        axes[0, 0].set_title('Box Loss')
        axes[0, 0].legend()
        axes[0, 0].grid(True)
    
    # Class Loss
    if 'train/cls_loss' in df.columns and 'val/cls_loss' in df.columns:
        axes[0, 1].plot(df['epoch'], df['train/cls_loss'], label='Train Class Loss', color='blue')
        axes[0, 1].plot(df['epoch'], df['val/cls_loss'], label='Val Class Loss', color='red')
        axes[0, 1].set_xlabel('Epoch')
        axes[0, 1].set_ylabel('Loss')
        axes[0, 1].set_title('Class Loss')
        axes[0, 1].legend()
        axes[0, 1].grid(True)
    
    # mAP50
    if 'metrics/mAP50(B)' in df.columns:
        axes[1, 0].plot(df['epoch'], df['metrics/mAP50(B)'], label='mAP50', color='green')
        axes[1, 0].set_xlabel('Epoch')
        axes[1, 0].set_ylabel('mAP50')
        axes[1, 0].set_title('mAP50')
        axes[1, 0].legend()
        axes[1, 0].grid(True)
    
    # mAP50-95
    if 'metrics/mAP50-95(B)' in df.columns:
        axes[1, 1].plot(df['epoch'], df['metrics/mAP50-95(B)'], label='mAP50-95', color='purple')
        axes[1, 1].set_xlabel('Epoch')
        axes[1, 1].set_ylabel('mAP50-95')
        axes[1, 1].set_title('mAP50-95')
        axes[1, 1].legend()
        axes[1, 1].grid(True)
    
    plt.tight_layout()
    
    # SAVE TO FILE - NO DISPLAY - ABSOLUTELY NO imshow
    plot_path = os.path.join(OUT_DIR, "training_curves.png")
    plt.savefig(plot_path, dpi=150, bbox_inches='tight', facecolor='white')
    plt.close(fig)  # CRITICAL: Close figure immediately
    del fig, axes, df  # Clean up memory
    
    print(f"✓ Training curves saved to: {plot_path}")
else:
    print("⚠ results.csv not found, skipping plot generation")



Generating training curves...
✓ Training curves saved to: /kaggle/working/outputs/training_curves.png


## 9. Optional: Grad-CAM Visualization (Saved Only, No Display)


In [20]:
# ============================================
# REAL Grad-CAM for YOLO11m-seg (SAVED ONLY)
# ============================================
import torch
import torch.nn.functional as F
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import cv2
from ultralytics import YOLO

class GradCAM:
    """
    Real Grad-CAM for Ultralytics YOLO11 Seg models.

    It:
    - hooks a conv layer (e.g. 'model.22')
    - runs a forward pass
    - backprops from the mean activation
    - produces a true activation-based heatmap
    - overlays it on the original image
    """

    def __init__(self, model, target_layer_name: str = None, img_size: int = None):
        self.model = model
        self.pytorch_model = model.model if hasattr(model, "model") else model
        self.target_layer_name = target_layer_name or "model.22"
        self.target_layer = None
        self.gradients = None
        self.activations = None
        self.hooks = []

        # infer image size from model if possible
        if hasattr(model, "overrides") and "imgsz" in model.overrides:
            self.img_size = int(model.overrides["imgsz"])
        else:
            self.img_size = img_size or 640

        self._register_hooks()
        self.pytorch_model.eval()

    def _register_hooks(self):
        """Register forward & backward hooks on the target conv layer."""

        def forward_hook(module, input, output):
            self.activations = output

        def backward_hook(module, grad_input, grad_output):
            if grad_output[0] is not None:
                self.gradients = grad_output[0]

        # try to find requested layer
        for name, module in self.pytorch_model.named_modules():
            if name == self.target_layer_name or self.target_layer_name in name:
                self.target_layer = module
                self.hooks.append(module.register_forward_hook(forward_hook))
                self.hooks.append(module.register_backward_hook(backward_hook))
                print(f"✓ Grad-CAM target layer: {name}")
                break

        # fallback = last Conv2d
        if self.target_layer is None:
            for name, module in reversed(list(self.pytorch_model.named_modules())):
                if isinstance(module, torch.nn.Conv2d):
                    self.target_layer = module
                    self.hooks.append(module.register_forward_hook(forward_hook))
                    self.hooks.append(module.register_backward_hook(backward_hook))
                    print(f"✓ Grad-CAM using fallback conv layer: {name}")
                    break

    def preprocess(self, img_array: np.ndarray) -> torch.Tensor:
        """
        Simple YOLO-like preprocessing:
        - resize to (img_size, img_size)
        - HWC -> CHW
        - [0,255] uint8 -> [0,1] float32
        - add batch dim
        """
        # ensure 3 channels
        if img_array.ndim == 2:
            img_array = np.stack([img_array] * 3, axis=-1)

        # resize to square
        img_resized = cv2.resize(img_array, (self.img_size, self.img_size))

        img = img_resized.astype(np.float32) / 255.0
        img = np.transpose(img, (2, 0, 1))  # HWC -> CHW
        img = np.ascontiguousarray(img)

        img_tensor = torch.from_numpy(img).unsqueeze(0)  # (1, C, H, W)

        device = next(self.pytorch_model.parameters()).device
        img_tensor = img_tensor.to(device)
        img_tensor.requires_grad_(True)
        return img_tensor

    def generate_cam(self, img_tensor: torch.Tensor, original_size):
        """
        Run forward + backward and compute CAM.
        original_size = (W, H)
        """
        self.gradients = None
        self.activations = None

        # forward pass
        outputs = self.pytorch_model(img_tensor)

        if self.activations is None:
            print("⚠ Grad-CAM: activations not captured, check target layer name.")
            return None

        # scalar target: mean activation (generic saliency)
        score = self.activations.mean()

        # backward
        self.pytorch_model.zero_grad(set_to_none=True)
        score.backward(retain_graph=False)

        if self.gradients is None:
            print("⚠ Grad-CAM: gradients not captured.")
            return None

        # compute weights and CAM
        grads = self.gradients       # (B, C, H, W)
        acts = self.activations      # (B, C, H, W)

        weights = grads.mean(dim=(2, 3), keepdim=True)  # (B, C, 1, 1)
        cam = (weights * acts).sum(dim=1)               # (B, H, W)

        cam = F.relu(cam)
        cam = cam[0].detach().cpu().numpy()

        # normalize to [0,1]
        cam -= cam.min() if cam.min() != cam.max() else 0
        if cam.max() > 0:
            cam /= cam.max()

        # resize CAM to original image size
        W, H = original_size
        cam_resized = cv2.resize(cam, (W, H))

        return cam_resized

    def visualize(self, image_path: str, detections: list, save_path: Path):
        """
        Create a figure with:
        - original image
        - original + Grad-CAM heatmap overlay
        - text info about detections

        Saves the figure to save_path, does not display in notebook.
        """
        # load original image (RGB)
        img = Image.open(image_path).convert("RGB")
        img_array = np.array(img)
        original_size = img.size  # (W, H)

        # preprocess & generate CAM
        img_tensor = self.preprocess(img_array)
        cam = self.generate_cam(img_tensor, original_size)

        if cam is None:
            print(f"⚠ Could not produce CAM for {image_path}")
            return

        # create overlay
        W, H = original_size
        cam_uint8 = np.uint8(255 * cam)

        # OpenCV expects BGR
        img_bgr = cv2.cvtColor(img_array, cv2.COLOR_RGB2BGR)
        heatmap = cv2.applyColorMap(cam_uint8, cv2.COLORMAP_JET)
        overlay_bgr = cv2.addWeighted(img_bgr, 0.5, heatmap, 0.5, 0)
        overlay_rgb = cv2.cvtColor(overlay_bgr, cv2.COLOR_BGR2RGB)

        # === Figure (no plt.show, only save) ===
        fig = plt.figure(figsize=(18, 6), facecolor="white")
        fig.suptitle("Grad-CAM Visualization", fontsize=14, fontweight="bold")

        gs = fig.add_gridspec(1, 3, hspace=0.3, wspace=0.3)

        # 1) Original image
        ax1 = fig.add_subplot(gs[0, 0])
        ax1.imshow(img_array)
        ax1.set_title("Original Image", fontsize=12, fontweight="bold")
        ax1.axis("off")

        # 2) Overlay
        ax2 = fig.add_subplot(gs[0, 1])
        ax2.imshow(overlay_rgb)
        ax2.set_title("Grad-CAM Overlay", fontsize=12, fontweight="bold")
        ax2.axis("off")

        # 3) Detection info
        ax3 = fig.add_subplot(gs[0, 2])
        det_info = f"Detections: {len(detections)}\n"
        for det in detections[:5]:
            det_info += f"Class {det['class']}: {det['confidence']:.2f}\n"
        ax3.text(0.5, 0.5, det_info, ha="center", va="center", fontsize=10)
        ax3.set_title("Detection Info", fontsize=12, fontweight="bold")
        ax3.axis("off")

        save_path.parent.mkdir(parents=True, exist_ok=True)
        plt.savefig(save_path, dpi=150, bbox_inches="tight", facecolor="white")
        plt.close(fig)

        print(f"✓ Grad-CAM saved: {save_path}")


# ============================
# RUN THE GENERATION PIPELINE
# ============================

print("=" * 100)
print(" Generating REAL Grad-CAM Visualizations ".center(100, "="))
print("=" * 100)
print()

# pick run_dir if not already defined
if "run_dir" not in globals():
    runs_base = Path("/kaggle/working/runs")
    if runs_base.exists():
        run_dirs = sorted(
            [d for d in runs_base.iterdir() if d.is_dir() and d.name.startswith("yolo11m_seg_")],
            key=lambda x: x.stat().st_mtime,
            reverse=True,
        )
        if run_dirs:
            run_dir = run_dirs[0]
            print(f"✓ Using most recent run directory: {run_dir}")
        else:
            print("⚠ No run directory found, skipping Grad-CAM")
            run_dir = None
    else:
        print("⚠ No runs directory found")
        run_dir = None
else:
    print(f"✓ Using run directory from training: {run_dir}")

if run_dir:
    val_images_dir = Path(f"{DATA_DIR}/images/val")
    if val_images_dir.exists():
        image_files = list(val_images_dir.glob("*.jpg"))[:4]  # a few samples

        if image_files:
            print("Generating REAL Grad-CAM visualizations (saved only)...")

            best_model_path = Path(run_dir) / "weights" / "best.pt"
            if best_model_path.exists():

                best_model = YOLO(str(best_model_path))
                gradcam = GradCAM(best_model, target_layer_name="model.22")

                gradcam_dir = Path(OUT_DIR) / "gradcam"
                gradcam_dir.mkdir(parents=True, exist_ok=True)

                for img_path in image_files:
                    # detections via high-level API (for info panel only)
                    pred_results = best_model.predict(
                        source=str(img_path),
                        conf=0.25,
                        save=False,
                        show=False,
                        verbose=False,
                    )

                    detections = []
                    boxes = pred_results[0].boxes
                    for i in range(len(boxes)):
                        detections.append(
                            {
                                "class": int(boxes.cls[i].item()),
                                "confidence": float(boxes.conf[i].item()),
                            }
                        )

                    save_path = gradcam_dir / f"{img_path.stem}_gradcam.png"
                    gradcam.visualize(str(img_path), detections, save_path)

                print(f"✓ Grad-CAM visualizations saved to: {gradcam_dir}")
            else:
                print("⚠ Best model weights not found")
    else:
        print("⚠ Validation images directory not found")

print()
print("=" * 100)


============================= Generating REAL Grad-CAM Visualizations ==============================

✓ Using run directory from training: /kaggle/working/runs/yolo11m_seg_pretrained_exact_1764499597
Generating REAL Grad-CAM visualizations (saved only)...
✓ Grad-CAM target layer: model.22
✓ Grad-CAM saved: /kaggle/working/outputs/gradcam/p_62933_gradcam.png
✓ Grad-CAM saved: /kaggle/working/outputs/gradcam/p_127581_gradcam.png
✓ Grad-CAM saved: /kaggle/working/outputs/gradcam/image_1021393_gradcam.png
✓ Grad-CAM saved: /kaggle/working/outputs/gradcam/pseudo_p_4141_gradcam.png
✓ Grad-CAM visualizations saved to: /kaggle/working/outputs/gradcam



In [19]:
# Optional: Run predictions on validation set (SAVED ONLY, NO DISPLAY)
ENABLE_PREDICTIONS = True  # Set to True to enable

if ENABLE_PREDICTIONS:
    val_images_dir = Path(f"{DATA_DIR}/images/val")
    if val_images_dir.exists():
        # Get a few sample images
        image_files = list(val_images_dir.glob("*.jpg"))[:5]  # Only 5 samples
        
        if image_files:
            print("Running sample predictions (saved only, no display)...")
            
            # Load best model
            best_model_path = run_dir / "weights" / "best.pt"
            if best_model_path.exists():
                best_model = YOLO(str(best_model_path))
                
                # Run predictions - CRITICAL: show=False, save=True
                pred_results = best_model.predict(
                    source=[str(img) for img in image_files],
                    conf=0.25,
                    save=True,
                    show=False,  # CRITICAL: No display
                    project=OUT_DIR,
                    name="sample_predictions",
                    verbose=False
                )
                
                print(f"✓ Sample predictions saved to: {OUT_DIR}/sample_predictions")
            else:
                print("⚠ Best model weights not found, skipping predictions")
        else:
            print("⚠ No validation images found")
    else:
        print("⚠ Validation images directory not found")
else:
    print("Sample predictions disabled (set ENABLE_PREDICTIONS = True to enable)")


Running sample predictions (saved only, no display)...
Results saved to /kaggle/working/outputs/sample_predictions
✓ Sample predictions saved to: /kaggle/working/outputs/sample_predictions
